In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import mlflow
import dagshub
from dotenv import load_dotenv
import joblib, os

In [51]:
load_dotenv()
dagshub.init(repo_owner='IbrahimFaye', repo_name='weather-agri', mlflow=True)
mlflow.set_experiment("besoin_irrigation")

DATA_PATH = "../data/irrigation_dataset.csv"
df = pd.read_csv(DATA_PATH)

Initialized MLflow to track repo "IbrahimFaye/weather-agri"

Repository IbrahimFaye/weather-agri initialized!

In [49]:
df["water_need"].describe()

count    78912.000000
mean         4.865345
std          2.827958
min          0.300000
25%          2.720405
50%          4.255870
75%          7.845316
max          9.000000
Name: water_need, dtype: float64

In [27]:
df.head()

,lat,lon,observation_time,temp_c,humidity,precipitation_mm,wind_speed,pressure,hour,day,month,temp_rolling_mean,humidity_rolling_mean,precip_rolling_sum,wind_rolling_mean,et0,cum_rain_3days,water_need,irrigation_label
0,14.727592,-16.911621,2022-01-01 00:00:00+00:00,24.0,34.0,0.0,12.9,1013.1,0,1,1,24.000000,34.0,0.0,12.900000,2.0,0.0,2.0,medium
1,14.727592,-16.911621,2022-01-01 01:00:00+00:00,22.7,32.0,0.0,12.0,1013.0,1,1,1,23.350000,33.0,0.0,12.450000,2.0,0.0,2.0,medium
2,14.727592,-16.911621,2022-01-01 02:00:00+00:00,22.4,24.0,0.0,11.8,1012.9,2,1,1,23.033333,30.0,0.0,12.233333,2.0,0.0,2.0,medium
3,14.727592,-16.911621,2022-01-01 03:00:00+00:00,22.7,19.0,0.0,13.0,1012.4,3,1,1,22.600000,25.0,0.0,12.266667,2.0,0.0,2.0,medium
4,14.727592,-16.911621,2022-01-01 04:00:00+00:00,22.7,17.0,0.0,14.8,1011.9,4,1,1,22.600000,20.0,0.0,13.200000,2.0,0.0,2.0,medium


In [29]:
print(df.describe())

                lat           lon        temp_c      humidity  \
count  78912.000000  78912.000000  78912.000000  78912.000000   
mean      14.985354    -16.127482     27.555796     51.388940   
std        1.134025      0.555957      5.120990     27.878398   
min       13.743409    -16.911621     13.700000      3.000000   
25%       13.743409    -16.911621     24.100000     25.000000   
50%       14.727592    -15.785126     26.700000     50.000000   
75%       16.485060    -15.685699     31.000000     77.000000   
max       16.485060    -15.685699     46.200000    100.000000   

       precipitation_mm    wind_speed      pressure         hour  \
count      78912.000000  78912.000000  78912.000000  78912.00000   
mean           0.060325     12.292882   1011.623372     11.50000   
std            0.519937      5.199194      2.112629      6.92223   
min            0.000000      0.000000   1002.300000      0.00000   
25%            0.000000      8.500000   1010.300000      5.75000   
50%   

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78912 entries, 0 to 78911
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   lat                    78912 non-null  float64
 1   lon                    78912 non-null  float64
 2   observation_time       78912 non-null  object 
 3   temp_c                 78912 non-null  float64
 4   humidity               78912 non-null  float64
 5   precipitation_mm       78912 non-null  float64
 6   wind_speed             78912 non-null  float64
 7   pressure               78912 non-null  float64
 8   hour                   78912 non-null  int64  
 9   day                    78912 non-null  int64  
 10  month                  78912 non-null  int64  
 11  temp_rolling_mean      78912 non-null  float64
 12  humidity_rolling_mean  78912 non-null  float64
 13  precip_rolling_sum     78912 non-null  float64
 14  wind_rolling_mean      78912 non-null  float64
 15  et

In [31]:
def calculate_realistic_et0(df):
    """
    Calcul ET₀  calibré pour l'Afrique de l’Ouest
    Inspiré du modèle Hargreaves modifié.
    """
    T = df["temp_c"]
    RH = df["humidity"]
    WS = df["wind_speed"]
    month = df["month"]

    # Amplitude thermique sur 24h
    Tmax = df["temp_c"].rolling(24, min_periods=1).max()
    Tmin = df["temp_c"].rolling(24, min_periods=1).min()
    deltaT = np.maximum(Tmax - Tmin, 4)

    # Facteur saisonnier : plus fort en saison sèche
    seasonal_factor = month.map({
        11: 1.4, 12: 1.4, 1: 1.4, 2: 1.4, 3: 1.3, 4: 1.2,
        5: 1.1, 6: 1.0, 7: 1.0, 8: 1.0, 9: 1.1, 10: 1.2
    }).fillna(1.0)

    et0 = (
        0.0023 * (T + 17.8)
        * np.sqrt(deltaT)
        * (1 - RH / 200)
        * (1 + WS / 20)
        * seasonal_factor
    )

    et0 = np.clip(et0, 2.0, 8.0)

    return et0.fillna(4.0)
    
def calculate_realistic_water_need(df):
    """
    Calcul réaliste du besoin en irrigation (mm/jour)
    """
    et0 = calculate_realistic_et0(df)
    effective_rain = df["precip_rolling_sum"] * 0.75  
    water_need = et0 - effective_rain
    return np.clip(water_need, 0.5, 9.0)


In [53]:
def prepare_realistic_training_data():
    df = pd.read_csv("../data/irrigation_dataset.csv")
    
    print("=== STATISTIQUES BESOINS EN EAU ===")
    print(f"Moyenne: {df['water_need'].mean():.2f} mm/jour")
    print(f"Min: {df['water_need'].min():.2f} mm/jour")
    print(f"Max: {df['water_need'].max():.2f} mm/jour")
    print(f"Médiane: {df['water_need'].median():.2f} mm/jour")
    
    assert df['water_need'].mean() > 1.0, "Les besoins en eau sont trop bas!"
    assert df['water_need'].max() < 15.0, "Les besoins en eau sont trop élevés!"
    
    return df

def train_realistic_irrigation_model():
    df = prepare_realistic_training_data()
    
    drop_cols = ["observation_time", "irrigation_label"]
    features = [
        "lat", "lon", "temp_c", "humidity", "precipitation_mm", "wind_speed",
        "pressure", "hour", "day", "month", "temp_rolling_mean", "humidity_rolling_mean",
        "precip_rolling_sum", "wind_rolling_mean", "et0", "cum_rain_3days"
    ]
    
    X = df[features]
    y = df["water_need"]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    with mlflow.start_run(run_name="Realistic_Irrigation_Model") as run:
        model = RandomForestRegressor(
            n_estimators=150,
            max_depth=15,
            min_samples_leaf=3,
            min_samples_split=10,
            n_jobs=-1,
            random_state=42
        )
        
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        mae = mean_absolute_error(y_test, preds)
        r2 = r2_score(y_test, preds)

        print(f"💧 MODÈLE RÉALISTE — MAE: {mae:.3f} mm/jour | R²: {r2:.3f}")
        print(f"💧 Plage des prédictions: {preds.min():.2f} - {preds.max():.2f} mm/jour")

        mlflow.log_metrics({"mae": mae, "r2": r2})
        
        os.makedirs("artifacts", exist_ok=True)
        joblib.dump(model, "artifacts/irrigation_model1.pkl", compress=3)
        mlflow.log_artifact("artifacts/irrigation_model1.pkl", artifact_path="model")

        importances = dict(zip(X.columns, model.feature_importances_))
        mlflow.log_dict(importances, "feature_importance.json")
        
        print("✅ Modèle  entraîné et sauvegardé!")

if __name__ == "__main__":
    df = prepare_realistic_training_data()
    print(df[["temp_c","humidity","wind_speed","et0","water_need"]].describe())
    train_realistic_irrigation_model()


=== STATISTIQUES BESOINS EN EAU ===
Moyenne: 4.87 mm/jour
Min: 0.30 mm/jour
Max: 9.00 mm/jour
Médiane: 4.26 mm/jour
             temp_c      humidity    wind_speed           et0    water_need
count  78912.000000  78912.000000  78912.000000  78912.000000  78912.000000
mean      27.555796     51.388940     12.292882      3.362273      4.865345
std        5.120990     27.878398      5.199194      1.902454      2.827958
min       13.700000      3.000000      0.000000      1.000000      0.300000
25%       24.100000     25.000000      8.500000      1.551262      2.720405
50%       26.700000     50.000000     12.000000      3.167876      4.255870
75%       31.000000     77.000000     15.800000      4.804232      7.845316
max       46.200000    100.000000     34.900000      7.000000      9.000000
=== STATISTIQUES BESOINS EN EAU ===
Moyenne: 4.87 mm/jour
Min: 0.30 mm/jour
Max: 9.00 mm/jour
Médiane: 4.26 mm/jour
💧 MODÈLE RÉALISTE — MAE: 0.005 mm/jour | R²: 1.000
💧 Plage des prédictions: 0.30 - 9